# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [2]:
import re
from collections import Counter

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg


nltk.download("gutenberg", raise_on_error=True)
documentos_ids = sorted(gutenberg.fileids())

def normalizar_texto(texto):
    """Aplicar la misma tokenización"""
    return re.findall(r"[^\W\d_]+", texto.lower(), flags=re.UNICODE)

originales = {doc_id: gutenberg.raw(doc_id) for doc_id in documentos_ids}
corpus = {doc_id: normalizar_texto(texto) for doc_id, texto in originales.items()}

def obtener_titulo(texto):
    encabezado = texto.splitlines()[0].strip().strip("[]")
    return re.split(r",?\s+by\s+", encabezado, maxsplit=1, flags=re.I)[0]

titulos = {doc_id: obtener_titulo(texto) for doc_id, texto in originales.items()}
longitudes = np.array([len(corpus[d]) for d in documentos_ids])
vocabulario = sorted({t for tokens in corpus.values() for t in tokens})
N = len(documentos_ids)

display(pd.DataFrame([
    {"Identificador": d, "Título": titulos[d], "Términos únicos": len(set(corpus[d]))}
    for d in documentos_ids
]))
print(f"Documentos: {N} | Vocabulario: {len(vocabulario)} términos únicos")
print(f"Longitud en tokens: mínima {longitudes.min()}, media {longitudes.mean():.2f}, máxima {longitudes.max()}")

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\sodom\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


,Identificador,Título,Términos únicos
0,austen-emma.txt,Emma,7096
1,austen-persuasion.txt,Persuasion,5740
2,austen-sense.txt,Sense and Sensibility,6283
3,bible-kjv.txt,The King James Bible,12568
4,blake-poems.txt,Poems,1510
5,bryant-stories.txt,Stories to Tell to Children,3834
6,burgess-busterbrown.txt,The Adventures of Buster Bear,1521
7,carroll-alice.txt,Alice's Adventures in Wonderland,2569
8,chesterton-ball.txt,The Ball and The Cross,8176
9,chesterton-brown.txt,The Wisdom of Father Brown,7737


Documentos: 18 | Vocabulario: 41509 términos únicos
Longitud en tokens: mínima 6934, media 118670.78, máxima 791842


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [3]:
def calcular_tf(tokens):
    return Counter(tokens)

def calcular_df(conteos):
    df = Counter()
    for conteo in conteos:
        df.update(conteo.keys())  # Cada término aporta 1 por documento.
    return df

def calcular_idf(n_documentos, frecuencia_documental):
    return np.log(n_documentos / frecuencia_documental) + 1

def calcular_peso(tf, idf):
    return tf * idf

conteos = {d: calcular_tf(corpus[d]) for d in documentos_ids}
df = calcular_df(conteos.values())
idf = {t: calcular_idf(N, df[t]) for t in vocabulario}

ejemplos = []
for d in ["melville-moby_dick.txt", "carroll-alice.txt"]:
    for t in ["whale", "rabbit", "the"]:
        tf = conteos[d][t]
        ejemplos.append([titulos[d], t, tf, df[t], idf[t], calcular_peso(tf, idf[t])])

display(pd.DataFrame(ejemplos, columns=["Libro", "Término", "TF", "DF", "IDF", "TF-IDF"]))

,Libro,Término,TF,DF,IDF,TF-IDF
0,Moby Dick,whale,1226,6,2.098612,2572.898666
1,Moby Dick,rabbit,0,6,2.098612,0.000000
2,Moby Dick,the,14431,18,1.000000,14431.000000
3,Alice's Adventures in Wonderland,whale,0,6,2.098612,0.000000
4,Alice's Adventures in Wonderland,rabbit,51,6,2.098612,107.029227
5,Alice's Adventures in Wonderland,the,1642,18,1.000000,1642.000000


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [4]:
columna = {t: j for j, t in enumerate(vocabulario)}
matriz_tf = np.zeros((N, len(vocabulario)), dtype=int)
for i, d in enumerate(documentos_ids):
    for t, frecuencia in conteos[d].items():
        matriz_tf[i, columna[t]] = frecuencia

vector_idf = np.array([idf[t] for t in vocabulario])
matriz_tfidf = calcular_peso(matriz_tf, vector_idf)
print("Dimensiones (documentos, términos):", matriz_tfidf.shape)
muestra = ["whale", "rabbit", "the"]
display(pd.DataFrame(matriz_tfidf[:, [columna[t] for t in muestra]],
                     index=documentos_ids, columns=muestra).round(3))

Dimensiones (documentos, términos): (18, 41509)


,whale,rabbit,the
austen-emma.txt,0.000,0.000,5204.0
austen-persuasion.txt,0.000,0.000,3329.0
austen-sense.txt,0.000,0.000,4105.0
bible-kjv.txt,6.296,0.000,64023.0
blake-poems.txt,0.000,0.000,439.0
bryant-stories.txt,33.578,20.986,3452.0
burgess-busterbrown.txt,0.000,23.085,660.0
carroll-alice.txt,0.000,107.029,1642.0
chesterton-ball.txt,4.197,8.394,4981.0
chesterton-brown.txt,0.000,6.296,4670.0


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

def usar_tokens(tokens):
    return tokens

vectorizador = TfidfVectorizer(
    analyzer=usar_tokens, vocabulary=columna, use_idf=True,
    smooth_idf=False, sublinear_tf=False, norm=None
)
matriz_sk = vectorizador.fit_transform([corpus[d] for d in documentos_ids])
tolerancia = 1e-10
assert vectorizador.get_feature_names_out().tolist() == vocabulario
diferencia_pesos = np.max(np.abs(matriz_tfidf - matriz_sk.toarray()))
print(f"Máxima diferencia absoluta entre pesos: {diferencia_pesos:.3e}")
assert diferencia_pesos <= tolerancia

Máxima diferencia absoluta entre pesos: 0.000e+00


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

consultas = {
    "whale harpoon sea": "Encontrar pasajes sobre la caza de ballenas en el mar.",
    "rabbit queen wonderland": "Localizar el relato de Alicia, el conejo y la reina.",
    "prophet covenant israel": "Encontrar textos bíblicos sobre profetas y la alianza de Israel.",
    "marriage family fortune": "Encontrar relatos que relacionen matrimonio, familia y fortuna."
}
textos_consulta = list(consultas)
tokens_consulta = [normalizar_texto(q) for q in textos_consulta]
display(pd.DataFrame(consultas.items(), columns=["Consulta", "Necesidad de información"]))

def vector_consulta(tokens):
    vector = np.zeros(len(vocabulario))
    for t, tf in calcular_tf(tokens).items():
        if t in columna:
            vector[columna[t]] = calcular_peso(tf, idf[t])
    return vector

def cosenos_explicitos(Q, D):
    productos = Q @ D.T
    denominadores = np.linalg.norm(Q, axis=1)[:, None] * np.linalg.norm(D, axis=1)
    return np.divide(productos, denominadores, out=np.zeros_like(productos),
                     where=denominadores != 0)

Q = np.stack([vector_consulta(tokens) for tokens in tokens_consulta])
Q_sk = vectorizador.transform(tokens_consulta)
similitudes = cosenos_explicitos(Q, matriz_tfidf)
similitudes_sk = cosine_similarity(Q_sk, matriz_sk)
np.testing.assert_allclose(Q, Q_sk.toarray(), atol=tolerancia, rtol=0)
diferencia_cosenos = np.max(np.abs(similitudes - similitudes_sk))
print(f"Máxima diferencia absoluta entre los {similitudes.size} cosenos: {diferencia_cosenos:.3e}")
assert diferencia_cosenos <= tolerancia

,Consulta,Necesidad de información
0,whale harpoon sea,Encontrar pasajes sobre la caza de ballenas en...
1,rabbit queen wonderland,"Localizar el relato de Alicia, el conejo y la ..."
2,prophet covenant israel,Encontrar textos bíblicos sobre profetas y la ...
3,marriage family fortune,"Encontrar relatos que relacionen matrimonio, f..."


Máxima diferencia absoluta entre los 72 cosenos: 1.860e-15


In [7]:
i = documentos_ids.index("melville-moby_dick.txt")
q, d = Q[0], matriz_tfidf[i]
compartidos = np.flatnonzero((q > 0) & (d > 0))
productos_parciales = q[compartidos] * d[compartidos]
print(f"{textos_consulta[0]} · {titulos[documentos_ids[i]]}")
display(pd.DataFrame({
    "Término compartido": [vocabulario[j] for j in compartidos],
    "Peso consulta": q[compartidos], "Peso documento": d[compartidos],
    "Producto parcial": productos_parciales
}))
print(f"Producto punto: {productos_parciales.sum():.6f}")
print(f"Norma de consulta: {np.linalg.norm(q):.6f} | Norma del libro: {np.linalg.norm(d):.6f}")
print(f"Similitud coseno: {similitudes[0, i]:.8f}")

whale harpoon sea · Moby Dick


,Término compartido,Peso consulta,Peso documento,Producto parcial
0,harpoon,3.197225,242.989068,776.890620
1,sea,1.117783,508.591281,568.494706
2,whale,2.098612,2572.898666,5399.516758


Producto punto: 6744.902084
Norma de consulta: 3.984452 | Norma del libro: 21539.783392
Similitud coseno: 0.07858972


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [8]:
def fragmento(doc_id, tokens):
    texto = originales[doc_id]
    # Mostrar el contexto del término de consulta más distintivo presente en el libro.
    presentes = sorted(set(tokens) & conteos[doc_id].keys(), key=lambda t: (-idf[t], t))
    inicio = 0
    if presentes:
        patron = r"(?<![^\W\d_])" + re.escape(presentes[0]) + r"(?![^\W\d_])"
        coincidencia = re.search(patron, texto, flags=re.IGNORECASE)
        if coincidencia:
            inicio = max(0, coincidencia.start() - 55)
    fin = min(len(texto), inicio + 165)
    pasaje = " ".join(texto[inicio:fin].split())
    return ("…" if inicio else "") + pasaje + ("…" if fin < len(texto) else "")

def top_cinco(puntuaciones):
    return sorted(range(N), key=lambda i: (-puntuaciones[i], documentos_ids[i]))[:5]

def tabla_ranking(puntuaciones, tokens):
    filas = []
    for puesto, i in enumerate(top_cinco(puntuaciones), start=1):
        d = documentos_ids[i]
        filas.append([puesto, d, titulos[d], puntuaciones[i], fragmento(d, tokens)])
    return pd.DataFrame(filas, columns=["Posición", "Identificador", "Título", "Puntuación", "Fragmento"])

for j, consulta in enumerate(textos_consulta):
    print(consulta)
    assert top_cinco(similitudes[j]) == top_cinco(similitudes_sk[j])
    with pd.option_context("display.max_colwidth", None):
        display(tabla_ranking(similitudes[j], tokens_consulta[j]))

whale harpoon sea


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick,0.078590,"…rder to discover when they were nigh enough to risk a harpoon from the bowsprit? Now having a night, a day, and still another night following before me in New…"
1,2,whitman-leaves.txt,Leaves of Grass,0.004886,"…m, The mate stands braced in the whale-boat, lance and harpoon are ready, The duck-shooter walks by silent and cautious stretches, The deacons are ordain'd with cros…"
2,3,bryant-stories.txt,Stories to Tell to Children,0.004622,"…him out again. HOW BROTHER RABBIT FOOLED THE WHALE AND THE ELEPHANT One day little Brother Rabbit was running along on the sand, lippety, lippety, wh…"
3,4,milton-paradise.txt,Paradise Lost,0.002690,"…Typhon, whom the den By ancient Tarsus held, or that sea-beast Leviathan, which God of all his works Created hugest that swim th' ocean-stream. Him, haply slumb…"
4,5,chesterton-brown.txt,The Wisdom of Father Brown,0.002638,"…d specialist in certain moral disorders, lay along the sea-front at Scarborough, in a series of very large and well-lighted french windows, which showed the North Se…"


rabbit queen wonderland


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,carroll-alice.txt,Alice's Adventures in Wonderland,0.029992,[Alice's Adventures in Wonderland by Lewis Carroll 1865] CHAPTER I. Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her sister on the bank…
1,2,burgess-busterbrown.txt,The Adventures of Buster Bear,0.006570,…elt that he had the greatest news to tell since Peter Rabbit had first found the tracks of Buster Bear in the Green Forest. He couldn't keep it to himself a minute…
2,3,bryant-stories.txt,Stories to Tell to Children,0.002527,"…ss somebody picked him out again. HOW BROTHER RABBIT FOOLED THE WHALE AND THE ELEPHANT One day little Brother Rabbit was running along on the sand, lip…"
3,4,blake-poems.txt,Poems,0.002433,"…t a dream! What can it mean? And that I was a maiden Queen Guarded by an Angel mild: Witless woe was ne'er beguiled! And I wept both night and day, And he wi…"
4,5,shakespeare-hamlet.txt,The Tragedie of Hamlet,0.001012,"…ou? Ham. Not so my Lord, I am too much i'th' Sun Queen. Good Hamlet cast thy nightly colour off, And let thine eye looke like a Friend on Denmarke. Do not for…"


prophet covenant israel


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,bible-kjv.txt,The King James Bible,0.039622,"…rth shall die. 6:18 But with thee will I establish my covenant; and thou shalt come into the ark, thou, and thy sons, and thy wife, and thy sons' wives with thee.…"
1,2,bryant-stories.txt,Stories to Tell to Children,0.002771,"…country were named Israelites. All the strong men of Israel went up to the battle, to fight for their king. David's three older brothers went, but he was only a b…"
2,3,milton-paradise.txt,Paradise Lost,0.002565,"…future days, As I shall thee enlighten; intermix My covenant in the Woman's seed renewed; So send them forth, though sorrowing, yet in peace: And on the east si…"
3,4,melville-moby_dick.txt,Moby Dick,0.001028,"…might well appal me. ""Will he the (leviathan) make a covenant with thee? Behold the hope of him is vain! But I have swam through libraries and sailed through oc…"
4,5,shakespeare-macbeth.txt,The Tragedie of Macbeth,0.000654,"…Name of King vpon me, And bad them speake to him. Then Prophet-like, They hayl'd him Father to a Line of Kings. Vpon my Head they plac'd a fruitlesse Crowne, And put…"


marriage family fortune


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,austen-persuasion.txt,Persuasion,0.013038,"…with an old country family of respectability and large fortune, and had therefore given all the honour and received none: Elizabeth would, one day or other, marry su…"
1,2,austen-sense.txt,Sense and Sensibility,0.011173,"…y respectable young man, was amply provided for by the fortune of his mother, which had been large, and half of which devolved on him on his coming of age. By his ow…"
2,3,austen-emma.txt,Emma,0.007676,"…r. Weston was a man of unexceptionable character, easy fortune, suitable age, and pleasant manners; and there was some satisfaction in considering with what self-den…"
3,4,chesterton-brown.txt,The Wisdom of Father Brown,0.004801,"…ambridge. He left me practically all his pretty large fortune; and I am sure he meant it in contempt. ""Arthur, you may say, might well complain of this; but A…"
4,5,shakespeare-hamlet.txt,The Tragedie of Hamlet,0.004794,"…aith, her priuates, we Ham. In the secret parts of Fortune? Oh, most true: she is a Strumpet. What's the newes? Rosin. None my Lord; but that the World's grown…"


### Comparar TF-IDF y BM25

In [9]:
def puntuar_bm25(tokens, k1=1.2, b=0.75):
    puntuaciones = np.zeros(N)
    ajuste_longitud = k1 * (1 - b + b * longitudes / longitudes.mean())
    for t in sorted(set(tokens)):
        if t not in columna:
            continue
        frecuencias = matriz_tf[:, columna[t]]
        idf_bm25 = np.log1p((N - df[t] + 0.5) / (df[t] + 0.5))
        puntuaciones += idf_bm25 * frecuencias * (k1 + 1) / (frecuencias + ajuste_longitud)
    return puntuaciones

for j, consulta in enumerate(textos_consulta):
    bm25 = puntuar_bm25(tokens_consulta[j])
    orden_tf = top_cinco(similitudes[j])
    orden_bm = top_cinco(bm25)
    print(f"{consulta} | Libros en común en el top-5: {len(set(orden_tf) & set(orden_bm))}/5")
    display(pd.DataFrame({
        "Posición": range(1, 6),
        "Libro TF-IDF": [titulos[documentos_ids[i]] for i in orden_tf],
        "Coseno": similitudes[j, orden_tf],
        "Libro BM25": [titulos[documentos_ids[i]] for i in orden_bm],
        "BM25": bm25[orden_bm]
    }))

whale harpoon sea | Libros en común en el top-5: 3/5


,Posición,Libro TF-IDF,Coseno,Libro BM25,BM25
0,1,Moby Dick,0.078590,Moby Dick,7.015046
1,2,Leaves of Grass,0.004886,Leaves of Grass,4.170131
2,3,Stories to Tell to Children,0.004622,Stories to Tell to Children,2.567654
3,4,Paradise Lost,0.002690,The Tragedie of Hamlet,2.150998
4,5,The Wisdom of Father Brown,0.002638,The Ball and The Cross,1.917374


rabbit queen wonderland | Libros en común en el top-5: 3/5


,Posición,Libro TF-IDF,Coseno,Libro BM25,BM25
0,1,Alice's Adventures in Wonderland,0.029992,Alice's Adventures in Wonderland,6.598005
1,2,The Adventures of Buster Bear,0.006570,The Man Who Was Thursday,4.234794
2,3,Stories to Tell to Children,0.002527,Stories to Tell to Children,2.618797
3,4,Poems,0.002433,The Ball and The Cross,2.279065
4,5,The Tragedie of Hamlet,0.001012,The Adventures of Buster Bear,2.272213


prophet covenant israel | Libros en común en el top-5: 4/5


,Posición,Libro TF-IDF,Coseno,Libro BM25,BM25
0,1,The King James Bible,0.039622,The King James Bible,8.123301
1,2,Stories to Tell to Children,0.002771,Paradise Lost,6.260923
2,3,Paradise Lost,0.002565,Moby Dick,4.627224
3,4,Moby Dick,0.001028,Stories to Tell to Children,2.574235
4,5,The Tragedie of Macbeth,0.000654,The Tragedie of Hamlet,1.783101


marriage family fortune | Libros en común en el top-5: 3/5


,Posición,Libro TF-IDF,Coseno,Libro BM25,BM25
0,1,Persuasion,0.013038,Sense and Sensibility,2.530000
1,2,Sense and Sensibility,0.011173,Persuasion,2.525673
2,3,Emma,0.007676,Emma,2.506258
3,4,The Wisdom of Father Brown,0.004801,The Man Who Was Thursday,2.168625
4,5,The Tragedie of Hamlet,0.004794,Leaves of Grass,2.132767


### Interpretación de los resultados

En *Moby Dick*, `whale` aparece 1226 veces. Como figura en 6 de los 18 documentos, su IDF es aproximadamente 2.0986 y su peso en ese libro es 2572.90. En *Alice's Adventures in Wonderland*, el IDF de `whale` es idéntico, pero su TF es cero y también lo es su peso.

La consulta `whale harpoon sea` sitúa *Moby Dick* primero con ambos métodos. Su coseno es 0.07859. Una consulta de tres palabras comparte solo una parte del vocabulario de un libro entero, por lo que una puntuación pequeña puede bastar para ocupar el primer puesto.

En `marriage family fortune`, el coseno sitúa primero *Persuasion* y luego *Sense and Sensibility*. BM25 invierte esos dos puestos, con puntuaciones de 2.530 para *Sense and Sensibility* y 2.526 para *Persuasion*. Las normas de los vectores, la saturación de TF y el ajuste por longitud son formas distintas de valorar las coincidencias. Determinar cuál orden responde mejor a la necesidad del usuario requeriría juicios de relevancia sobre los libros.